# Sensor budgets for a target error

Paper §4.5: VND + Beam and Greedy budget searches. Returned counts are verified feasible budgets, not certified global minima.


In [ ]:
from pathlib import Path
import sys, importlib, importlib.util
from IPython.display import display, Markdown

candidates = []
for parent in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    candidates.extend([parent, parent/'problem-2-dual', parent/'problem-2-min-error-seq/problem-2-dual'])
HERE = next(p for p in candidates if (p/'dual_search.py').is_file())
sys.path.insert(0, str(HERE))
import dual_search
importlib.reload(dual_search)
import greedy_comparison
importlib.reload(greedy_comparison)
import budget_limited
importlib.reload(budget_limited)
spec = importlib.util.spec_from_file_location('dual_reporting', HERE/'reporting.py')
dual_reporting = importlib.util.module_from_spec(spec)
spec.loader.exec_module(dual_reporting)

RUN_MODE = 'full'  # 'full' or 'compact'; saved reports use their own metadata.
RUN_ANALYSIS = False
SAVED_RESULTS = HERE/'results'/RUN_MODE
METHODS = ('vnd_beam', 'greedy')  # Select either or both; other method caches remain available.
RESUME = True
TARGET_ERRORS = (0.1, 0.2, 0.3, 0.4, 0.5)
BEAM_WIDTH = 3
MAX_ITERATIONS = 30
TIME_LIMIT_SECONDS = None  # Optional positive per-budget heuristic time limit.
BOOTSTRAP_RESAMPLES = 2000
BOOTSTRAP_SEED = 20261015

MAX_SENSOR_BOUND = 5000


## Budget search

Uses existing extended inputs and sequential perfect-discrimination bounds. See §4.5 for the search; interrupted searches retain verified feasible solutions and can resume.


In [ ]:
if RUN_ANALYSIS:
    REPORT_MODE = RUN_MODE
    REPORT_TARGET_ERRORS = TARGET_ERRORS
    solutions, failures, OUTPUT = budget_limited.run_comparison(
        RUN_MODE, targets=TARGET_ERRORS, beam_width=BEAM_WIDTH, methods=METHODS,
        max_sensor_bound=MAX_SENSOR_BOUND,
        max_iterations=MAX_ITERATIONS, time_limit=TIME_LIMIT_SECONDS,
        resume=RESUME, run=True,
    )
else:
    import json
    OUTPUT = Path(SAVED_RESULTS)
    saved_info = json.loads((OUTPUT/'run_info.json').read_text())
    REPORT_MODE = saved_info['mode']
    REPORT_TARGET_ERRORS = tuple(saved_info['targets'])
    print(f'Loading saved {REPORT_MODE} results; no searches will run.')
    if REPORT_MODE != RUN_MODE or REPORT_TARGET_ERRORS != TARGET_ERRORS:
        print('RUN_MODE and TARGET_ERRORS apply only when RUN_ANALYSIS=True.')
    solutions = json.loads((OUTPUT/'solutions.json').read_text())
    failures = json.loads((OUTPUT/'failures.json').read_text())
import json
skipped_path = OUTPUT/'skipped.json'
skipped = json.loads(skipped_path.read_text()) if skipped_path.exists() else []
print('Skipped instances:', len(skipped))
assert all(r['verified_feasible'] and r['actual_error'] <= r['epsilon'] for r in solutions)
assert all(sum(r['allocation']) == r['N'] <= r['N_min_seq'] for r in solutions)
print('Report mode:', REPORT_MODE)
print('Report targets:', REPORT_TARGET_ERRORS)
print('Verified solutions:', len(solutions))
print('Failed instances:', len(failures))
print('Results:', OUTPUT)

if not any(r.get('method') == 'greedy' for r in solutions):
    print('Greedy results are not saved yet. Set RUN_ANALYSIS=True to compute them; matching VND results are reused.')


## Sensor counts versus target error

Paired completed cases; curves show medians and bands show interquartile ranges. Runtime includes perfect-allocation setup and target-specific search.


In [ ]:
# Reload labels and styling when rerunning this plotting cell.
import importlib.util
spec = importlib.util.spec_from_file_location('dual_reporting', HERE / 'reporting.py')
dual_reporting = importlib.util.module_from_spec(spec)
spec.loader.exec_module(dual_reporting)

import matplotlib.pyplot as plt

figure, summary, coverage = dual_reporting.report(
    solutions, failures, OUTPUT, REPORT_TARGET_ERRORS,
    resamples=BOOTSTRAP_RESAMPLES, seed=BOOTSTRAP_SEED, skipped=skipped,
)
display(figure)
plt.close(figure)

def show_table(rows):
    if not rows:
        print('No rows available')
        return
    columns = list(rows[0])
    lines = ['| ' + ' | '.join(columns) + ' |', '| ' + ' | '.join(['---']*len(columns)) + ' |']
    for row in rows:
        lines.append('| ' + ' | '.join(f'{row[k]:.6g}' if isinstance(row[k],float) else str(row[k]) for k in columns) + ' |')
    display(Markdown('\n'.join(lines)))

display(Markdown('### Target-error summary'))
show_table(summary)
display(Markdown('### Coverage'))
show_table(coverage)
if failures:
    display(Markdown('### Failed instances (first 20; all retained in failures.json)'))
    show_table(failures[:20])

if skipped:
    display(Markdown("### Instances excluded by the sensor-bound cutoff"))
    show_table(skipped)


## Outputs

`results/{full,compact}/` contains solutions, summaries and figures. Set `RUN_ANALYSIS=False` and select `SAVED_RESULTS` to rebuild reports without searches.
